---

## **Projet Deep Learning**
## **Classification de tissus cancéreux colorectaux**

---

---

**Ce notebook est conçu pour être :**
- **reproductible** (chemins relatifs, seeds fixées)
- **idempotent** (relançable sans retélécharger si les fichiers sont déjà présents)
- **traçable** (quality gates go/no-go explicites)

---

---

# PARTIE 5 : ViT Vision Transformer FROM SCRATCH

---

---

## Plan du notebook

| Cellule | Section | Contenu |
|---------|---------|---------|
| 1 | — | Header
| 2 | — | Principes, contexte clinique, objectif, question de départ |
| 3 | — | Plan du notebook |
| 4 | 1. Configuration | Imports |
| 5 | 1. Configuration | Versions (traçabilité) |
| 6 | 1. Configuration | Seed, device |
| 7 | 1. Configuration | Chemins relatifs |
| 8 | 2. Acquisition | Téléchargement PathMNIST (idempotent) |
| 9 | 3. Validation technique | Métadonnées complètes |
| 10 | 3. Validation technique | Quality gate go/no-go |
| 11 | 4. Structure des données | Carte d'identité (dtype, pixels, canaux RGB, labels) |
| 12 | 4. Structure des données | Shapes des 3 splits + élément individuel |
| 13 | 5. Visualisation | Un exemple par classe |
| 14 | 5. Visualisation | Analyse : premières observations visuelles |
| 15 | 5. Visualisation | 5 exemples par classe (variabilité intra-classe, seedé) |
| 16 | 5. Visualisation | Analyse : ce qu'on observe par classe |
| 17 | 6. Distribution | Histogrammes des 3 splits + tableau domain shift |
| 18 | 6. Distribution | Analyse : observation train vs test (domain shift) |
| 19 | 6. Distribution | Boxplots par classe et par split |
| 20 | 6. Distribution | Stats descriptives complètes (moy, std, quartiles, skew, kurtosis) |
| 21 | 7. Q1.1 | Comparaison visuelle Debris vs Background (5 exemples seedés + asserts) |
| 22 | 7. Q1.1 | Analyse Q1.1 |
| 23 | 8. Q1.2 | Réponse stricte : 1 image + stats RGB + comparaison ImageNet |
| 24 | 8. Q1.2 | Analyse Q1.2 |
| 25 | Bonus | Comparaison globale PathMNIST vs ImageNet (tout le dataset) |
| 26 | Bonus | Comparaison unitaire sain vs cancer (images + canaux + tableau) |
| 27 | Bonus | Analyse : sain vs cancer |
| 28 | Bonus | Barres mean/std : adipose vs cancer vs ImageNet (2x2) |
| 29 | Bonus | Histogrammes distribution pixels : sain vs cancer |
| 30 | Bonus | Analyse : ce que les histogrammes nous apprennent |
| 31 | 9. Décisions figées | Code : NORM_MEAN, NORM_STD, constantes |
| 32 | 9. Décisions figées | Pourquoi normaliser les données |
| 33 | 9. Décisions figées | Contrat complet pour la modélisation |
| 34 | 9. Décisions figées | Sanity check final |

---

---

### Objectif

Construire un Vision Transformer (ViT) from scratch — pas de poids pré-entraînés. Le ViT découpe l'image en patches et utilise le mécanisme d'attention (comme GPT) pour apprendre les relations entre ces patches.

### Progression du projet
- **Partie 2 (MLP)** : 63.08% — pixels isolés
- **Partie 3 (CNN)** : 92.38% — zones locales (filtres 3×3)
- **Partie 4 (ResNet-18)** : en cours — connaissances pré-apprises
- **Partie 5 (ViT)** : relations globales entre zones de l'image via l'attention

### Consignes du sujet
- Patch embedding + CLS token + positional embeddings
- `nn.TransformerEncoder` pour la self-attention
- Classification via CLS token
- Justifier les hyperparamètres
- **Q5.1** : Patch size 7 vs 14 — nombre de patches, pourquoi 14 est mauvais, lancer les deux
- **Q5.2** : Retirer positional embeddings, mesurer la chute d'accuracy, expliquer pourquoi
- **Q5.3** : Comparer nb paramètres ViT vs CNN, expliquer pourquoi ViT peut sous-performer

### Rappels techniques (Lab 3)
- Le Lab 3 partie 1 (sequence reversal) donne la structure : embedding, positional encoding, `nn.TransformerEncoder`
- Le ViT est entraîné from scratch (pas de pré-entraînement) — il devra tout apprendre sur nos 90 000 images

---

---

### Choix d'architecture et d'hyperparamètres ViT — justifications

**Pourquoi un Vision Transformer :**
- Le CNN balaie l'image avec des filtres locaux (3×3) — il voit bien les détails mais a une vision limitée du contexte global
- Le ViT découpe l'image en patches et laisse chaque patch "regarder" tous les autres grâce à l'attention — il a une vision globale dès la première couche
- En théorie, cette vision globale pourrait aider à distinguer des tissus qui se différencient par leur organisation spatiale à grande échelle (stroma vs muscle par exemple)

**Patch size 7 (choix principal) :**
- L'image fait 28×28, un patch de 7×7 donne 4×4 = 16 patches
- 16 patches est un bon compromis : assez de granularité pour capter les textures, assez peu pour que le mécanisme d'attention soit efficace
- Chaque patch de 7×7×3 = 147 pixels est projeté dans un espace d'embedding

**Patch size 14 (comparaison Q5.1) :**
- 28÷14 = 2×2 = 4 patches seulement
- Trop peu : chaque patch couvre un quart de l'image, le modèle perd toute notion de texture locale
- Le mécanisme d'attention n'a que 4 éléments à comparer — pas assez pour apprendre des relations pertinentes

**CLS token :**
- Un token spécial ajouté au début de la séquence de patches
- Il n'est lié à aucune zone de l'image — son rôle est de résumer l'image entière via l'attention
- C'est son embedding final qui est utilisé pour la classification (comme le [CLS] de BERT en NLP)

**Positional embeddings :**
- Les patches arrivent au Transformer comme une séquence, mais sans information sur leur position dans l'image
- Les positional embeddings sont des vecteurs apprenables qui encodent "ce patch est en haut à gauche, celui-ci est en bas à droite"
- Sans eux, le Transformer traiterait l'image comme un ensemble non ordonné de patches — il ne saurait pas que deux patches voisins sont liés (Q5.2)

**nn.TransformerEncoder :**
- On utilise l'implémentation PyTorch standard (même approche que le Lab 3)
- Self-attention : chaque patch "regarde" tous les autres pour comprendre le contexte global de l'image
- On peut empiler plusieurs couches d'attention pour des représentations plus profondes

**Pas de pré-entraînement — conséquences attendues :**
- Contrairement au ResNet (partie 4), le ViT part de poids aléatoires
- Les Transformers ont besoin de beaucoup de données pour bien apprendre (ViT original entraîné sur 300M images)
- Avec 90 000 images, le ViT pourrait sous-performer par rapport au CNN car il n'a pas de biais inductif spatial (le CNN "sait" que les pixels voisins sont liés grâce aux convolutions, le ViT doit l'apprendre) (Q5.3)
- C'est un trade-off : plus de flexibilité (attention globale) mais besoin de plus de données

**CrossEntropyLoss + Adam (lr=0.001, weight_decay=1e-4) :**
- Cohérence avec les parties précédentes
- Weight decay = régularisation L2 (pénalise les poids trop grands, réduit l'overfitting)

---


In [1]:
%matplotlib inline
print("=== C7 — Imports ===")

# Imports
# Version 1.0

import sys
import os
import time
import copy
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import medmnist
import seaborn as sns
from medmnist import PathMNIST, INFO
from torchvision import transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix, classification_report
from scipy import stats as sp_stats
print("Imports OK")



In [2]:
print("=== C8 — Versions — traçabilité ===")
# Versions — traçabilité
print(f"Python   : {sys.version.split()[0]}")
print(f"PyTorch  : {torch.__version__}")
print(f"MedMNIST : {medmnist.__version__}")
print(f"NumPy    : {np.__version__}")



In [3]:
print("=== C9 — Reproductibilité complète (CPU + GPU + cuDNN) ===")
# Reproductibilité complète (CPU + GPU + cuDNN)
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device        : {device}")
print(f"cuDNN determ. : {torch.backends.cudnn.deterministic if torch.cuda.is_available() else 'N/A'}")



In [4]:
print("=== C10 — Constantes et dataset — calculées dans l'EDA (note ===")
# Constantes et dataset — calculées dans l'EDA (notebook 1)
DATA_DIR = os.path.join(".", "data")
# Valeurs à mettre à jour après run NB1 (méthode DataLoader)
NORM_MEAN = [0.7405, 0.5330, 0.7058]
NORM_STD  = [0.1237, 0.1768, 0.1244]
N_CLASSES = 9

# Recharger le dataset
train_dataset = PathMNIST(split='train', download=False, root=DATA_DIR)
val_dataset   = PathMNIST(split='val',   download=False, root=DATA_DIR)
test_dataset  = PathMNIST(split='test',  download=False, root=DATA_DIR)

info = train_dataset.info
labels_names = info['label']
CLASS_NAMES = list(labels_names.values())

# Timer global du notebook
notebook_start_time = time.time()

print(f"NORM_MEAN : {NORM_MEAN}")
print(f"NORM_STD  : {NORM_STD}")
print(f"Train : {len(train_dataset)} | Val : {len(val_dataset)} | Test : {len(test_dataset)}")
print("✓ Lien avec notebook 1 établi")

# Objectif : réduire les faux négatifs cancer (cancer non détecté)
train_labels_all = train_dataset.labels.flatten()
class_counts = np.bincount(train_labels_all)
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum() * len(class_weights)
# Surpondérer cancer (classe 8) et mucosa (classe 6)
class_weights[8] *= 2.0  # cancer : un faux négatif peut coûter une vie
class_weights[6] *= 1.5  # mucosa : souvent confondu avec cancer
CLASS_WEIGHTS = torch.FloatTensor(class_weights).to(device)
print(f"Class weights : {[round(w, 4) for w in class_weights.tolist()]}")





In [5]:
print("=== C11 — Fonction d'entraînement — comme dans le Lab 1 ===")
# Fonction d'entraînement — comme dans le Lab 1
# Entraîne le modèle et enregistre les métriques à chaque époque
# Retourne un historique (dict) pour tracer les courbes d'apprentissage

def train_model(model, train_loader, val_loader, epochs=50, learning_rate=0.001, weight_decay=1e-4):
    """
    Boucle d'entraînement complète.
    - CrossEntropyLoss : loss standard pour la classification multi-classe
    - Adam : optimiseur adaptatif (ajuste le learning rate automatiquement)
    - On évalue sur le val à chaque époque pour surveiller l'overfitting
    - On sauvegarde le meilleur modèle (val_loss minimale)
    """
    criterion = nn.CrossEntropyLoss()
    # weight_decay : régularisation L2 pour empêcher les poids de devenir trop gros
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    # Scheduler : baisse progressivement le learning rate (cosinus) pour affiner la convergence
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    history = {
        'loss': [], 'accuracy': [],
        'val_loss': [], 'val_accuracy': []
    }
    
    best_val_loss = float('inf')
    best_model_state = None
    best_epoch = 0
    
    for epoch in range(epochs):
        # --- Phase TRAIN ---
        model.train()
        train_loss, train_correct, train_total = 0, 0, 0
        
        for images, labels_batch in train_loader:
            images = images.to(device)
            labels_batch = labels_batch.squeeze().long().to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels_batch)
            loss.backward()
            # Gradient clipping — stabilise l entraînement des Transformers (Lab 3)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            train_loss += loss.item() * images.size(0)
            train_correct += (outputs.argmax(1) == labels_batch).sum().item()
            train_total += images.size(0)
        
        # --- Phase VALIDATION ---
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        
        with torch.no_grad():
            for images, labels_batch in val_loader:
                images = images.to(device)
                labels_batch = labels_batch.squeeze().long().to(device)
                
                outputs = model(images)
                loss = criterion(outputs, labels_batch)
                
                val_loss += loss.item() * images.size(0)
                val_correct += (outputs.argmax(1) == labels_batch).sum().item()
                val_total += images.size(0)
        
        epoch_train_loss = train_loss / train_total
        epoch_train_acc = train_correct / train_total
        epoch_val_loss = val_loss / val_total
        epoch_val_acc = val_correct / val_total
        
        history['loss'].append(epoch_train_loss)
        history['accuracy'].append(epoch_train_acc)
        history['val_loss'].append(epoch_val_loss)
        history['val_accuracy'].append(epoch_val_acc)
        
        # Sauvegarde du meilleur modèle
        scheduler.step()

        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch + 1
        
        print(f"Epoch {epoch+1:>3}/{epochs} | "
              f"Train loss: {epoch_train_loss:.6f} acc: {epoch_train_acc:.6f} | "
              f"Val loss: {epoch_val_loss:.6f} acc: {epoch_val_acc:.6f}"
              f"{' ← best' if epoch + 1 == best_epoch else ''}")
    
    # Restaurer le meilleur modèle (EN DEHORS de la boucle)
    model.load_state_dict(best_model_state)
    print(f"\n✓ Meilleur modèle restauré (époque {best_epoch}, val_loss: {best_val_loss:.6f})")
    
    return history

import time

print("✓ Fonction train_model chargée")









In [6]:
print("=== C12 — Preprocessing ViT — même que CNN (images 3×28×28,  ===")
# Preprocessing ViT — même que CNN (images 3×28×28, normalisées)
# Le ViT travaille sur les mêmes images que le CNN, pas de resize nécessaire

vit_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(NORM_MEAN, NORM_STD)
])

train_vit = PathMNIST(split='train', transform=vit_transform, download=False, root=DATA_DIR)
val_vit   = PathMNIST(split='val',   transform=vit_transform, download=False, root=DATA_DIR)
test_vit  = PathMNIST(split='test',  transform=vit_transform, download=False, root=DATA_DIR)

BATCH_SIZE = 64
train_loader_vit = DataLoader(train_vit, batch_size=BATCH_SIZE, shuffle=True)
val_loader_vit   = DataLoader(val_vit,   batch_size=BATCH_SIZE, shuffle=False)
test_loader_vit  = DataLoader(test_vit,  batch_size=BATCH_SIZE, shuffle=False)

# Vérification
sample_img, sample_label = train_vit[0]
assert sample_img.shape == (3, 28, 28), f"Shape attendue (3,28,28), obtenue {sample_img.shape}"
print(f"Image ViT : shape={sample_img.shape}, dtype={sample_img.dtype}")
print(f"DataLoaders ViT créés (batch_size={BATCH_SIZE})")
print("✓ Preprocessing ViT terminé")



In [7]:
print("=== C13 — Vérification sur un batch complet ===")
# Vérification sur un batch complet
batch_imgs, batch_labels = next(iter(train_loader_vit))
print(f"Batch shape  : {batch_imgs.shape}")
print(f"Batch mean   : {batch_imgs.mean():.4f}")
print(f"Batch std    : {batch_imgs.std():.4f}")
assert batch_imgs.shape[1:] == (3, 28, 28), f"Shape attendue (3,28,28), obtenue {batch_imgs.shape[1:]}"
print("✓ Batch ViT vérifié")



In [8]:
# Création du modèle ViT — Vision Transformer from scratch
# Inspiré du Lab 3 (TransformerEncoder) adapté aux images

class VisionTransformer(nn.Module):
    def __init__(self, img_size=28, patch_size=7, in_channels=3, n_classes=9,
                 embed_dim=128, n_heads=4, n_layers=4, dropout=0.1, use_pos_embed=True):
        super().__init__()
        self.patch_size = patch_size
        self.use_pos_embed = use_pos_embed

        # Nombre de patches : (28/7)² = 16
        n_patches = (img_size // patch_size) ** 2
        patch_dim = in_channels * patch_size * patch_size  # 3 × 7 × 7 = 147

        # Patch embedding : projeter chaque patch dans l'espace d'embedding
        self.patch_embed = nn.Linear(patch_dim, embed_dim)

        # CLS token : token spécial pour la classification
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))

        # Positional embeddings : apprendre la position de chaque patch
        if use_pos_embed:
            self.pos_embed = nn.Parameter(torch.randn(1, n_patches + 1, embed_dim))

        # Transformer Encoder (comme dans le Lab 3)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=n_heads,
            dim_feedforward=embed_dim * 4,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        # Classification head : CLS token → classes
        self.mlp_head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, n_classes)
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B = x.shape[0]  # batch size

        # Découper l'image en patches
        # (B, 3, 28, 28) → (B, n_patches, patch_dim)
        p = self.patch_size
        patches = x.unfold(2, p, p).unfold(3, p, p)  # (B, 3, 4, 4, 7, 7)
        patches = patches.contiguous().view(B, -1, 3 * p * p)  # (B, 16, 147)

        # Projeter dans l'espace d'embedding
        x = self.patch_embed(patches)  # (B, 16, embed_dim)

        # Ajouter le CLS token
        cls_tokens = self.cls_token.expand(B, -1, -1)  # (B, 1, embed_dim)
        x = torch.cat([cls_tokens, x], dim=1)  # (B, 17, embed_dim)

        # Ajouter les positional embeddings
        if self.use_pos_embed:
            x = x + self.pos_embed

        x = self.dropout(x)

        # Passer dans le Transformer
        x = self.transformer(x)  # (B, 17, embed_dim)

        # Classification via le CLS token (premier token)
        cls_output = x[:, 0]  # (B, embed_dim)

        return self.mlp_head(cls_output)  # (B, n_classes)

# Q5.3 — Comparer le nombre de paramètres
print("=== Création ViT (patch_size=7) ===")
torch.manual_seed(SEED)
vit_model = VisionTransformer(patch_size=7)
vit_model = vit_model.to(device)

total_params = sum(p.numel() for p in vit_model.parameters())
trainable_params = sum(p.numel() for p in vit_model.parameters() if p.requires_grad)
print(f"Nb paramètres total       : {total_params:,}")
print(f"Nb paramètres entraînables : {trainable_params:,}")
print("✓ ViT créé (patch_size=7)")


=== Création ViT (patch_size=7) ===
Nb paramètres total       : 815,753
Nb paramètres entraînables : 815,753
✓ ViT créé (patch_size=7)


---

## Expérience 1 : ViT avec patch_size=7

---


In [9]:
# Entraînement ViT (patch_size=7) — 50 époques
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

start_time = time.time()
print("=== Entraînement ViT patch_size=7 (50 époques) ===")
print(f"Entraînement sur {device}...")
history_vit_7 = train_model(vit_model, train_loader_vit, val_loader_vit, epochs=50, learning_rate=0.001)
train_time_vit_7 = time.time() - start_time

print(f"\nTemps d'entraînement : {train_time_vit_7:.1f}s ({train_time_vit_7/60:.1f} min)")




---

### Résultats ViT patch_size=7 (50 époques)

Le ViT atteint 86.1% de val accuracy — correct mais nettement en dessous du CNN (98.7%) et du ResNet (99.3%). Le modèle continue de progresser à l'époque 50 (best epoch 48), il n'a pas encore convergé.

Le train accuracy (85.2%) est proche du val (86.1%) — pas d'overfitting. Le ViT apprend lentement mais régulièrement, sans mémoriser le train.

C'est un résultat attendu : le ViT part de zéro (pas de pré-entraînement) et n'a aucun a priori spatial (contrairement au CNN qui sait que les pixels voisins sont liés). Avec 90 000 images, il n'a pas assez de données pour tout apprendre seul.

---


In [10]:
print("=== C18 — Courbes d'entraînement ViT patch_size=7 ===")
# Courbes d'entraînement ViT patch_size=7
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, len(history_vit_7['accuracy']) + 1)

ax1.plot(epochs_range, history_vit_7['accuracy'], label='Train')
ax1.plot(epochs_range, history_vit_7['val_accuracy'], label='Val')
ax1.set_xlabel('Époque')
ax1.set_ylabel('Accuracy')
ax1.set_title('Accuracy par époque')
ax1.legend()

ax2.plot(epochs_range, history_vit_7['loss'], label='Train')
ax2.plot(epochs_range, history_vit_7['val_loss'], label='Val')
ax2.set_xlabel('Époque')
ax2.set_ylabel('Loss')
ax2.set_title('Loss par époque')
ax2.legend()

plt.suptitle("Courbes d'entraînement — ViT patch_size=7 (50 époques)", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'NB5_vit_curves_patch7.png'), dpi=120, bbox_inches='tight')
plt.show()



In [11]:
# Évaluation ViT patch_size=7 sur le test set
vit_model.eval()
test_correct, test_total = 0, 0
all_preds_vit = []
all_labels_vit = []

with torch.no_grad():
    for images, labels_batch in test_loader_vit:
        images = images.to(device)
        labels_batch = labels_batch.squeeze().long().to(device)

        outputs = vit_model(images)
        preds = outputs.argmax(1)

        test_correct += (preds == labels_batch).sum().item()
        test_total += images.size(0)

        all_preds_vit.extend(preds.cpu().numpy())
        all_labels_vit.extend(labels_batch.cpu().numpy())

test_accuracy_vit = test_correct / test_total

print("=== Résultats ViT patch_size=7 (test set) ===")
print(f"  Test accuracy : {test_accuracy_vit:.6f}")


=== Résultats ViT patch_size=7 (test set) ===
  Test accuracy : 0.788719


---

### Résultats ViT patch_size=7 — test set

**Test accuracy : 0.788719 (78.87%)** — en dessous du val (86.1%), le domain shift frappe encore. Le ViT est plus sensible au changement d'hôpital que le CNN.

Le ViT fait mieux que le MLP (63.08%) mais moins bien que le CNN (92.38%) et le ResNet (91.53%). Sa vision globale (attention entre tous les patches) ne compense pas son manque de biais inductif spatial et de données.

---


In [12]:
print("=== C21 — Matrice de confusion ViT patch_size=7 ===")
# Matrice de confusion ViT patch_size=7
cm_vit = confusion_matrix(all_labels_vit, all_preds_vit)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm_vit, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
ax.set_xlabel('Prédit')
ax.set_ylabel('Vrai')
ax.set_title('Matrice de confusion — ViT patch_size=7 (test set)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'NB5_vit_confusion_patch7.png'), dpi=120, bbox_inches='tight')
plt.show()

cm_vit_off = cm_vit.copy()
np.fill_diagonal(cm_vit_off, 0)
max_idx = np.unravel_index(cm_vit_off.argmax(), cm_vit_off.shape)
max_count = cm_vit_off[max_idx]
print(f"\nPlus grande confusion : {max_count} images de '{CLASS_NAMES[max_idx[0]]}' classées comme '{CLASS_NAMES[max_idx[1]]}'")

print("\n=== Classification report (test set) ===")
print(classification_report(all_labels_vit, all_preds_vit, target_names=CLASS_NAMES, digits=4))
# Tableau texte de la matrice de confusion
print("
=== Matrice de confusion (valeurs) ===")
print(f"{'':>35s}", end='')
for name in CLASS_NAMES:
    print(f"{name[:8]:>9s}", end='')
print()
print("-" * (35 + 9 * len(CLASS_NAMES)))
for i, row in enumerate(cm_vit.tolist()):
    print(f"{CLASS_NAMES[i]:<35s}", end='')
    for val in row:
        print(f"{val:>9d}", end='')
    print()




In [ ]:
print("=== C22 — BONUS — Visualisation des attention maps (Lab 3, E ===")
# BONUS — Visualisation des attention maps (Lab 3, Exercise 1.3/2.4)
# Extraire les poids d attention du ViT pour voir quelles zones de l image le modèle regarde
# C est le pendant du Grad-CAM (NB6) mais pour les Transformers

# TODO : implémenter après le run
# 1. Hook sur le module d attention du premier TransformerEncoderLayer
# 2. Forward pass sur une image test
# 3. Reshape les attention weights en grille 4×4 (16 patches)
# 4. Superposer sur l image originale
print("Attention maps : à implémenter après le run")



---

## Q5.1 — Comparaison patch_size=7 vs patch_size=14

---


In [13]:
# Q5.1 — ViT avec patch_size=14 (2×2 = 4 patches seulement)
print("=== Création ViT (patch_size=14) ===")
torch.manual_seed(SEED)
vit_model_14 = VisionTransformer(patch_size=14)
vit_model_14 = vit_model_14.to(device)

total_params_14 = sum(p.numel() for p in vit_model_14.parameters())
print(f"Nb paramètres : {total_params_14:,}")
print(f"Nb patches    : {(28//14)**2} (vs {(28//7)**2} pour patch_size=7)")
print("✓ ViT créé (patch_size=14)")

# Entraînement
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

start_time = time.time()
print("\n=== Entraînement ViT patch_size=14 (50 époques) ===")
history_vit_14 = train_model(vit_model_14,  train_loader_vit, val_loader_vit, epochs=50, learning_rate=0.001)
train_time_vit_14 = time.time() - start_time
print(f"\nTemps d'entraînement : {train_time_vit_14:.1f}s ({train_time_vit_14/60:.1f} min)")




In [14]:
# Évaluation ViT patch_size=14 sur le test set
vit_model_14.eval()
test_correct_14, test_total_14 = 0, 0

with torch.no_grad():
    for images, labels_batch in test_loader_vit:
        images = images.to(device)
        labels_batch = labels_batch.squeeze().long().to(device)
        outputs = vit_model_14(images)
        preds = outputs.argmax(1)
        test_correct_14 += (preds == labels_batch).sum().item()
        test_total_14 += images.size(0)

test_accuracy_vit_14 = test_correct_14 / test_total_14

print("=== Q5.1 — Comparaison patch_size ===")
print(f"  patch_size=7  : {test_accuracy_vit:.6f} ({(28//7)**2} patches)")
print(f"  patch_size=14 : {test_accuracy_vit_14:.6f} ({(28//14)**2} patches)")
print(f"  Différence    : {(test_accuracy_vit - test_accuracy_vit_14)*100:+.2f} pts")


=== Q5.1 — Comparaison patch_size ===
  patch_size=7  : 0.788719 (16 patches)
  patch_size=14 : 0.687604 (4 patches)
  Différence    : +10.11 pts


---

### Q5.1 — Analyse : patch_size=7 vs patch_size=14

| Configuration | Nb patches | Test accuracy |
|---|---|---|
| patch_size=7 | 16 (grille 4×4) | 78.87% |
| patch_size=14 | 4 (grille 2×2) | 68.76% |
| **Différence** | | **+10.11 pts** |

Avec seulement 4 patches, le modèle découpe l'image en 4 gros blocs. Chaque bloc couvre un quart de l'image — trop grossier pour capter les textures qui distinguent les types de tissus. Le mécanisme d'attention n'a que 4 éléments à comparer, ce qui limite sa capacité à trouver des relations pertinentes.

À 16 patches, chaque patch de 7×7 pixels capture une zone assez fine pour distinguer des textures locales tout en laissant à l'attention suffisamment d'éléments pour construire une vision globale.

---


---

## Q5.2 — Impact des positional embeddings

---


In [15]:
# Q5.2 — ViT SANS positional embeddings
print("=== Création ViT (patch_size=7, SANS positional embeddings) ===")
torch.manual_seed(SEED)
vit_model_no_pos = VisionTransformer(patch_size=7, use_pos_embed=False)
vit_model_no_pos = vit_model_no_pos.to(device)

# Entraînement
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

start_time = time.time()
print("\n=== Entraînement ViT SANS pos embed (50 époques) ===")
history_vit_no_pos = train_model(vit_model_no_pos,  train_loader_vit, val_loader_vit, epochs=50, learning_rate=0.001)
train_time_no_pos = time.time() - start_time
print(f"\nTemps d'entraînement : {train_time_no_pos:.1f}s ({train_time_no_pos/60:.1f} min)")




In [16]:
# Évaluation ViT sans pos embed
vit_model_no_pos.eval()
test_correct_np, test_total_np = 0, 0

with torch.no_grad():
    for images, labels_batch in test_loader_vit:
        images = images.to(device)
        labels_batch = labels_batch.squeeze().long().to(device)
        outputs = vit_model_no_pos(images)
        preds = outputs.argmax(1)
        test_correct_np += (preds == labels_batch).sum().item()
        test_total_np += images.size(0)

test_accuracy_no_pos = test_correct_np / test_total_np

print("=== Q5.2 — Impact positional embeddings ===")
print(f"  Avec pos embed : {test_accuracy_vit:.6f}")
print(f"  Sans pos embed : {test_accuracy_no_pos:.6f}")
print(f"  Chute          : {(test_accuracy_vit - test_accuracy_no_pos)*100:+.2f} pts")


=== Q5.2 — Impact positional embeddings ===
  Avec pos embed : 0.788719
  Sans pos embed : 0.807242
  Chute          : -1.85 pts


---

### Q5.2 — Analyse : impact des positional embeddings

| Configuration | Test accuracy |
|---|---|
| Avec positional embeddings | 78.87% |
| Sans positional embeddings | 80.72% |
| **Chute attendue** | **+1.85 pts (amélioration !)** |

Résultat surprenant : le ViT fait **mieux** sans les positional embeddings. Plusieurs explications :

1. **L'histologie n'a pas de position fixe** — un tissu cancéreux peut être n'importe où sur la lame. La position d'un patch dans l'image n'apporte pas d'information utile pour le diagnostic.

2. **16 patches c'est peu** — avec seulement 16 positions à encoder, les positional embeddings ajoutent des paramètres (17 × 128 = 2 176) qui créent du bruit sur un petit dataset plutôt que d'aider.

3. **Le contenu compte plus que la position** — ce qui distingue un tissu cancéreux d'un tissu sain, c'est sa couleur et sa texture, pas son emplacement dans l'image.

Ce résultat illustre une différence entre le NLP (où l'ordre des mots est crucial) et la vision (où la position spatiale n'est pas toujours pertinente, surtout en histologie).

---


In [ ]:
print("=== C31 — Sauvegarde du meilleur modèle ViT ===")
# Sauvegarde du meilleur modèle ViT
torch.save(vit_model.state_dict(), os.path.join(DATA_DIR, 'vit_model.pth'))

vit_results = {
    'model_name': 'ViT',
    'architecture': f'patch_size=7, embed_dim=128, n_heads=4, n_layers=4',
    'n_params': sum(p.numel() for p in vit_model.parameters()),
    'test_accuracy': test_accuracy_vit,
    'test_accuracy_patch14': test_accuracy_vit_14,
    'test_accuracy_no_pos': test_accuracy_no_pos,
    'train_time': train_time_vit_7,
}

print(f"Modèle ViT sauvegardé")
print(f"Test accuracy (patch 7)  : {test_accuracy_vit:.6f}")
print(f"Test accuracy (patch 14) : {test_accuracy_vit_14:.6f}")
print(f"Test accuracy (sans pos) : {test_accuracy_no_pos:.6f}")
print(f"Nb paramètres            : {vit_results['n_params']:,}")
print("✓ Sauvegarde terminée")

# Sauvegarder l'historique pour la partie 7 (Q7.1)
import pickle
with open(os.path.join(DATA_DIR, 'vit_history.pkl'), 'wb') as f:
    pickle.dump(history_vit_7, f)
print("✓ Historique sauvegardé")

# Sauvegarder les prédictions pour le recall dans NB7
with open(os.path.join(DATA_DIR, 'vit_preds.pkl'), 'wb') as f:
    pickle.dump({'all_preds': all_preds_vit, 'all_labels': all_labels_vit}, f)
print("✓ Prédictions ViT sauvegardées")





---

## Bilan Partie 5 — Vision Transformer

### Résultats

| Configuration | Val accuracy | Test accuracy | Temps | Nb paramètres |
|---|---|---|---|---|
| ViT patch_size=7 (avec pos) | 86.1% | 78.87% | 15.5 min | ~1.3M |
| ViT patch_size=14 (Q5.1) | 74.8% | 68.76% | 16.3 min | ~870K |
| ViT sans pos embed (Q5.2) | 87.2% | 80.72% | 15.3 min | ~1.3M |

### Comparaison avec les autres modèles

| Modèle | Test accuracy | Nb paramètres |
|---|---|---|
| MLP (partie 2) | 63.08% | 1.37M |
| CNN (partie 3) | 92.38% | ~280K |
| ResNet-18 frozen (partie 4) | 86.99% | 11.1M |
| ResNet-18 fine-tuning (partie 4) | 91.53% | 11.1M |
| **ViT patch_size=7** | **78.87%** | **~1.3M** |

### Ce qu'on a appris

1. **Le ViT from scratch sous-performe** — 78.87% vs 92.38% pour le CNN. Sans pré-entraînement et avec seulement 90K images, le Transformer n'a pas assez de données pour apprendre les features visuelles que le CNN capture naturellement grâce à ses convolutions.

2. **Plus de patches = mieux** (Q5.1) — patch_size=7 (16 patches) bat patch_size=14 (4 patches) de 10 points. La granularité des patches est essentielle.

3. **Les positional embeddings ne sont pas toujours utiles** (Q5.2) — en histologie, le contenu d'un patch compte plus que sa position. Le ViT sans positional embeddings fait même légèrement mieux (+1.85 pts).

4. **Le ViT a besoin de beaucoup de données** (Q5.3) — le ViT original a été entraîné sur 300M d'images. Avec 90K, il ne peut pas rivaliser. C'est pourquoi le CNN (qui a un biais inductif spatial) reste plus adapté aux petits datasets.

---


---

### Q5.3 — Comparaison nb paramètres ViT vs CNN

| Modèle | Paramètres | Test accuracy |
|--------|-----------|--------------|
| CNN (v1) | 436 649 | 91.78% |
| ViT (patch 7) | ~870 000 | à confirmer |
| ViT (patch 14) | ~870 000 | à confirmer |

Le ViT a 2× plus de paramètres que le CNN mais fait moins bien. Pourquoi :

- **Pas de biais inductif spatial** : le CNN sait que les pixels voisins sont liés grâce aux convolutions. Le ViT doit apprendre ça de zéro via l attention — il lui faut beaucoup plus de données
- **Dataset trop petit** : le ViT original a été entraîné sur 300M images. Avec 90K images, le ViT n a pas assez d exemples pour apprendre les relations spatiales que le CNN obtient gratuitement
- **Résolution trop basse** : à 28×28, les patches de 7×7 contiennent très peu d information. Le mécanisme d attention n a pas assez de matière pour extraire des patterns pertinents

*(Chiffres à mettre à jour après le run)*

---


In [17]:
print("=== C34 — Temps total d'exécution du notebook ===")
# Temps total d'exécution du notebook
notebook_total_time = time.time() - notebook_start_time
print(f"Temps total du notebook : {notebook_total_time:.1f}s ({notebook_total_time/60:.1f} min)")

